# Data inspection
Load the data which should be in `data/raw/all_data.csv` and show some common info.

In [208]:
import pandas as pd

df = pd.read_csv("data/raw/all_data.csv")

# We only care about english reviews
df = df[df["Review_Language"] == "English"]

print("\nColumns:")
print(df.columns.tolist())

print("\nShape:")
print(df.shape)

print("\nDescriptive stats:")
print(df.describe(include="all"))

print("\nHead:")
print(df.head())


Columns:
['Unnamed: 0', 'Review_Language', 'City Name', 'Hotel Name', 'Upside_Review', 'Downside_Review', 'Review_Score', 'Sentiment', 'source', 'Prompt_Language', 'na_up_review', 'na_down_review']

Shape:
(2000, 12)

Descriptive stats:
         Unnamed: 0 Review_Language City Name    Hotel Name Upside_Review  \
count   2000.000000            2000      2000          2000          1923   
unique          NaN               1        10           100          1847   
top             NaN         English    Ankara  Hotel Ickale      Location   
freq            NaN            2000       200            20            30   
mean     499.500000             NaN       NaN           NaN           NaN   
std      288.747186             NaN       NaN           NaN           NaN   
min        0.000000             NaN       NaN           NaN           NaN   
25%      249.750000             NaN       NaN           NaN           NaN   
50%      499.500000             NaN       NaN           NaN          

### Statistics about real- vs AI-generated reviews

In [209]:
# The source column refers to a real review (0) or AI-generated review (1)
print("\nSource value counts, real (0), AI-generated (1):")
source_counts = df["source"].value_counts()

print(source_counts)

avg_review_score = (
    df.groupby("source")["Review_Score"]
      .mean()
)

print("\nAverage review score by source:")
print(avg_review_score)


Source value counts, real (0), AI-generated (1):
source
0    1000
1    1000
Name: count, dtype: int64

Average review score by source:
source
0    6.554
1    5.706
Name: Review_Score, dtype: float64


# Results inspection
Load the `.txt` files results which should be ordered per `model` in the `./results` folder.

In [210]:
# load "results" directory and print models
import os


for filename in os.listdir("./results/"):
    print(filename)

decision_tree
random_forest
gradient_boosting
logistic_regression
naive_bayes


### Dataframe definition

Define massive dataframe which includes all results and their parameters.

In [211]:
# Per model, load the .txt file results
# Make a dataframe with the results for each model for each .txt file
df_results = pd.DataFrame()
for model_name in os.listdir("./results/"):
    model_path = os.path.join("./results/", model_name)
    if os.path.isdir(model_path):
        for result_file in os.listdir(model_path):
            if result_file.endswith(".txt"):
                result_path = os.path.join(model_path, result_file)
                with open(result_path, "r") as f:
                    lines = f.readlines()
                        
                    # Put those results in a df
                    df = pd.DataFrame()
                    
                    for i, line in enumerate(lines):
                        if "Results:" in line:
                            df.loc[0, "model"] = model_name
                            for j in range(i + 1, i + 5):
                                key, value = lines[j].strip().split(": ")
                                df.loc[0, key] = float(value)
                            break
                        
                        if "Parameters:" in line:
                            while True:
                                i += 1
                                if lines[i].strip() == "":
                                    break
                                key, value = lines[i].strip().split(": ")
                                df.loc[0, key] = value
                        
                        
                    # Add the results to the main dataframe
                    df_results = pd.concat([df_results, df], ignore_index=True)
 
# Columns to put first
first_columns = ["model", "Accuracy", "Precision", "Recall", "F1 Score"]
remaining_columns = [col for col in df_results.columns if col not in first_columns]

df_results = df_results[first_columns + remaining_columns]

df_results.head(5)

,model,Accuracy,Precision,Recall,F1 Score,Use stop words,Top N words,Number of features,ccp_alpha,class_weight,...,Chi2,TF-IDF,Lemmatization,N-grams,C,max_iter,alpha,class_prior,fit_prior,force_alpha
0,decision_tree,0.812,0.7910,0.848,0.8185,True,500,500,0.0020753462603878116,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,decision_tree,0.792,0.7808,0.812,0.7961,True,500,500,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,decision_tree,0.812,0.7910,0.848,0.8185,True,500,500,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,decision_tree,0.812,0.7910,0.848,0.8185,True,500,500,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,decision_tree,0.816,0.8319,0.792,0.8115,True,500,500,0.0,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Results per specified model
Display results for the `selected_model` and then drop columns which are not important to this model (e.g. logistic regression does not have the `min_samples_leaf` column). Finally display the results by *accuracy*.

In [212]:
selected_model = "logistic_regression"  # "decision_tree"  "logistic_regression", "random_forest", "gradient_boosting", "naive_bayes"

# Filter results for the selected model
df_model = df_results[df_results["model"] == selected_model].copy()

# Drop columns that contain only NaN values
# This is useful because some models may not have certain metrics or parameters, and we want to focus on the available data.
df_model = df_model.dropna(axis=1, how="all")
df_model = df_model.sort_values(by="F1 Score", ascending=False)

df_model

,model,Accuracy,Precision,Recall,F1 Score,Use stop words,Top N words,Number of features,Min doc frequency,Max doc frequency,Chi2,TF-IDF,Lemmatization,N-grams,C,max_iter
46,logistic_regression,0.940,0.9400,0.940,0.9400,True,2000,2000,1,1.0,True,True,False,"(1, 1)",10,250
56,logistic_regression,0.938,0.9433,0.932,0.9376,True,2500,2500,1,1.0,True,True,False,"(1, 1)",10,250
52,logistic_regression,0.938,0.9469,0.928,0.9374,True,1500,1500,1,1.0,True,True,False,"(1, 1)",10,250
48,logistic_regression,0.936,0.9467,0.924,0.9352,True,3000,3000,1,1.0,True,True,False,"(1, 1)",10,250
45,logistic_regression,0.934,0.9357,0.932,0.9339,True,1000,1000,1,1.0,True,True,False,"(1, 1)",10,250
55,logistic_regression,0.930,0.9352,0.924,0.9296,True,500,500,1,1.0,True,True,False,"(1, 1)",10,250
42,logistic_regression,0.926,0.9383,0.912,0.9249,True,2000,2000,1,1.0,True,True,True,"(1, 1)",10,250
50,logistic_regression,0.924,0.9274,0.920,0.9237,True,500,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43,logistic_regression,0.924,0.9274,0.920,0.9237,True,500,500,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN
32,logistic_regression,0.924,0.9274,0.920,0.9237,True,500,500,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN


In [213]:
import plotly.express as px

# Choose columns
x_col = "n_estimators"
y_col = "Accuracy"

# Keep NaN values as a visible category
df_plot = df_model.copy()
df_plot[x_col] = df_plot[x_col].fillna("NaN")

fig = px.scatter(
    df_plot,
    x=x_col,
    y=y_col,
    hover_data=df_plot.columns,
    title=f"{y_col} vs. {x_col}"
)

fig.show()

KeyError: 'n_estimators'